# Bokeh Notebook Tutorial

### *How to install:*

In [1]:
uv install bokeh

/Users/evijonas/python/archyviz/bin/python3: No module named uv
Note: you may need to restart the kernel to use updated packages.


#### OR

In [2]:
uv pip install bokeh

/Users/evijonas/python/archyviz/bin/python3: No module named uv
Note: you may need to restart the kernel to use updated packages.


#### ALSO

In [3]:
uv pip install bokeh_sampledata

/Users/evijonas/python/archyviz/bin/python3: No module named uv
Note: you may need to restart the kernel to use updated packages.


##### *(Should work for both Mac and Window set-ups)*

###### *Do this inside your environment via terminal*

### Bokeh Introduciton

##### Bokeh is an interactive visualization library that allows users to create browser based plots with built in tools such as zooming, panning, hovering, and selection. Unlike Matplotlib, which primarily produces static figures, Bokeh generates dynamic, responsive visualizations. This makes it especially useful for exploratory data analysis and interactive dashboards.

##### Bokeh renders plots in the browser using JavaScript while Python handles the data and plot configuration. When working in Jupyter Notebook or JupyterLab, Bokeh must be initialized with:

contour zoom in zoom out tool, You generate a 2D scalar field (z) on a polar grid, convert it to x/y coordinates, then ask Bokeh to draw contour bands (like a topographic map).

## Basic Line Chart

##### Exercise One Assets
 - figure(): Simple command to create your plot
 - .line(): graphs your data on the plot

In [4]:
from bokeh.plotting import figure, show, output_notebook
output_notebook()

x = [2, 4, 6, 8, 10]
y = [1, 3, 5, 7, 9]

p = figure(title="Basic Line Chart", x_axis_label='x', y_axis_label='y')

p.line(x, y, legend_label="Example Data", line_width=2)

show(p)

Loading BokehJS ...

## A Little More

In [5]:
import pandas as pd

from bokeh.palettes import Dark2  
from bokeh.plotting import output_notebook
from bokeh.sampledata.stocks import AAPL, GOOG, IBM, MSFT

p = figure(width=800, height=250, x_axis_type="datetime")
p.title.text = 'Click on stock entry to hide the corresponding lines'

for data, name, color in zip([AAPL, IBM, MSFT, GOOG], ["AAPL", "IBM", "MSFT", "GOOG"], Dark2[4]):
    df = pd.DataFrame(data)   
    df['date'] = pd.to_datetime(df['date'])
    p.line(df['date'], df['close'], line_width=2, color=color, alpha=0.8, legend_label=name)

p.legend.location = "top_left"
p.legend.click_policy="hide"

output_notebook()
show(p)

Loading BokehJS ...

## Exercise Three

In [6]:
import numpy as np
from bokeh.palettes import Magma256

# Data to contour is a 2D sin wave on a polar grid. 
radius, angle=np.meshgrid(np.linspace(0,1,20), np.linspace(0,2*np.pi,120))
x=radius*np.cos(angle)
y=radius*np.sin(angle)
z=1+np.sin(3*angle)*np.sin(np.pi*radius)

p = figure(width=550, height=400, match_aspect=True)  # keep circle round 
p.grid.grid_line_alpha = 0.25

levels = np.linspace(0, 2, 11) 
#Creates 11 contour thresholds → meaning the field gets split into 10 bands. More levels = more bands (more detail).

# Pink/Purple palette  
palette = Magma256 

#Bokeh computes the contour lines/bands based on z and the levels you give it, then draws based on later code
contour_renderer = p.contour(
    x=x, y=y, z=z,levels=levels,
    fill_color=palette,                      # fun colors 
    hatch_pattern=["x"]*5 + [" "]*5,        
    hatch_color="white",
    hatch_alpha=0.45,
    line_color=["white"]*5 + ["black"] + ["hotpink"]*5,  # pop lines 
    line_dash=["solid"]*6 + ["dashed"]*5,
    line_width=[1]*6 + [2]*5,
)

#A contour renderer knows its own color mapping, so Bokeh can build a matching colorbar automatically.
colorbar = contour_renderer.construct_color_bar(title="Pink/Purple scale")  
p.add_layout(colorbar, "above")

output_notebook()
show(p)

Loading BokehJS ...

## Exercise 4 (Periodic Table)

##### Assets
- dodge: applies a fixed offset to data (prevents overlap)

In [7]:
from bokeh.sampledata.periodic_table import elements
from bokeh.transform import dodge, factor_cmap

periods = ["I", "II", "III", "IV", "V", "VI", "VII"]
groups = [str(x) for x in range(1, 19)]

df = elements.copy()
df["atomic mass"] = df["atomic mass"].astype(str)
df["group"] = df["group"].astype(str)
df["period"] = [periods[x-1] for x in df.period]
df = df[df.group != "-"]
df = df[df.symbol != "Lr"]
df = df[df.symbol != "Lu"]

cmap = {
    "alkali metal"         : "#2e4f69",
    "alkaline earth metal" : "#2e7b69",
    "metal"                : "#6f7b9c",
    "halogen"              : "#b57baa",
    "metalloid"            : "#b5c6dd",
    "noble gas"            : "#a7beb1",
    "nonmetal"             : "#aebef6",
    "transition metal"     : "#c37ea1",
}

Hover = [
    ("Name", "@name"),
    ("Atomic number", "@{atomic number}"),
    ("Atomic mass", "@{atomic mass}"),
    ("Type", "@metal"),
    ("Electronic configuration", "@{electronic configuration}"),
]

p = figure(title="The Periodic Table", width=1000, height=450,
           x_range=groups, y_range=list(reversed(periods)),
           tools="hover", toolbar_location=None, tooltips=Hover)

r = p.rect("group", "period", 0.95, 0.95, source=df, fill_alpha=0.6, legend_field="metal",
           color=factor_cmap('metal', palette=list(cmap.values()), factors=list(cmap.keys())))

text_props = dict(source=df, text_align='left', text_baseline='middle')

x = dodge("group", -0.4, range=p.x_range)

p.text(x=x, y="period", text="symbol", text_font_style="bold", **text_props)

p.text(x=x, y=dodge("period", 0.3, range=p.y_range), text="atomic number",
       text_font_size="11px", **text_props)

p.text(x=x, y=dodge("period", -0.35, range=p.y_range), text="name",
       text_font_size="7px", **text_props)

p.text(x=x, y=dodge("period", -0.2, range=p.y_range), text="atomic mass",
       text_font_size="7px", **text_props)

p.text(x=["3", "3"], y=["VI", "VII"], text=["LA", "AC"], text_align="center", text_baseline="middle")

p.outline_line_color = None
p.grid.grid_line_color = None
p.axis.axis_line_color = None
p.axis.major_tick_line_color = None
p.axis.major_label_standoff = 0
p.legend.orientation = "horizontal"
p.legend.location ="top_center"
p.hover.renderers=[r]

output_notebook()
show(p)

Loading BokehJS ...

## Exercise Five

##### Assets:

- RangeTool: You make two plots, a main time series and a smaller “overview” plot below it, then you use RangeTool in the bottom plot to control the x range of the top plot.

In [11]:
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, RangeTool

# Data 
dates = np.array(AAPL['date'], dtype=np.datetime64)
source = ColumnDataSource(data=dict(date=dates, close=AAPL['adj_close']))
#Required for linked interactions. Both plots share the same source, so they stay consistent.

# Colors 
bg = "#f2f6ef"          # pale green-gray background 
gridc = "#d7e3d0"       # soft grid 
line_main = "#1b7f3a"   # dark green 
line_select = "#4caf50" # lighter green 
accent_y = "#c9d93a"    # yellow-green accent 

# Main figure 
p = figure(height=300, width=800, tools="xpan,xwheel_zoom,reset",
           x_axis_type="datetime", x_axis_location="above",
           background_fill_color=bg, border_fill_color=bg,
           x_range=(dates[1500], dates[2500]))
#You’re initially zooming in to a slice of the timeline (otherwise you’d see everything).

p.line('date', 'close', source=source, line_width=2.5, line_color=line_main)  # main line 
p.yaxis.axis_label = "Price"

p.grid.grid_line_color = gridc
p.grid.grid_line_alpha = 0.8
p.outline_line_color = None  

# Selector figure 
select = figure(title="Drag the middle and edges of the selection box to change the range above",
                height=130, width=800,
                x_axis_type="datetime", y_axis_type=None,
                tools="", toolbar_location=None,
                background_fill_color=bg, border_fill_color=bg)

select.line('date', 'close', source=source, line_width=2, line_color=line_select)  # mini line 
select.ygrid.grid_line_color = None
select.grid.grid_line_color = gridc
select.grid.grid_line_alpha = 0.7
select.outline_line_color = None

select.x_range.range_padding = 0
select.x_range.bounds = "auto"

# RangeTool (selection overlay) This is the “link.” The range tool’s selection window literally edits p.x_range live.
#Overlay styling
range_tool = RangeTool(x_range=p.x_range, start_gesture='pan')
range_tool.overlay.fill_color = accent_y      # yellow green overlay 
range_tool.overlay.fill_alpha = 0.25
range_tool.overlay.line_color = "#2f6f2f"     # edge color 
range_tool.overlay.line_dash = "dashed"
range_tool.overlay.line_width = 2


select.add_tools(range_tool)

output_notebook()
show(column(p, select))

Loading BokehJS ...

## Exercise 6 (Color Slider)

#### Assets
- column: arrangement in a single column
- ColumnDataSource: centralized source that maps strings to data sequences
- CustomJS: brings in JavaScript code
- curdoc: for retrieval of your "Document", allowing modification

In [18]:
from bokeh.models import CustomJS, Slider
from bokeh.plotting import curdoc
from bokeh.themes import Theme

R, G, B = (75, 125, 125)
color = f"#{R:x}{G:x}{B:x}"
text_color = "#ffffff"

# create a data source to enable refreshing of fill & text color
source = ColumnDataSource(data=dict(color=[color], text_color=[text_color]))

# create first plot, as a rect() glyph and centered text label, with fill and text color taken from source
p = figure(x_range=(-8, 8), y_range=(-4, 4),
           width=400, height=300,
           title='move sliders to change', tools='')

p.rect(0, 0, width=18, height=10, fill_color='color',
        line_color = 'black', source=source)

p.text(0, 0, text='color', text_color='text_color',
       alpha=0.6667, text_font_size='48px', text_baseline='middle',
       text_align='center', source=source)

red = Slider(title='red', start=0, end=255, value=R, step=1)
green = Slider(title="G", start=0, end=255, value=G, step=1)
blue = Slider(title="B", start=0, end=255, value=B, step=1)

callback = CustomJS(args=dict(source=source, red=red, blue=blue, green=green), code="""
    function toHex(c) {
        const hex = c.toString(16)
        return hex.length == 1 ? "0" + hex : hex
    }

    const R = red.value | 0
    const G = green.value | 0
    const B = blue.value | 0

    const color = "#" + toHex(R) + toHex(G) + toHex(B)
    const text_color = ((R > 127) || (G > 127) || (B > 127)) ? '#000000' : '#ffffff'
    source.data={color:[color], text_color:[text_color]}
""")
red.js_on_change('value', callback)
blue.js_on_change('value', callback)
green.js_on_change('value', callback)

curdoc().theme = Theme(json={
    "attrs": {
        "Plot": { "toolbar_location": None },
        "Grid": { "grid_line_color": None },
        "Axis": {
            "axis_line_color": None,
            "major_label_text_color": None,
            "major_tick_line_color": None,
            "minor_tick_line_color": None,
        },
    },
})
output_notebook()
show(column(red, green, blue,p))

Loading BokehJS ...

## Exercise Seven 

#### Assets:

- lasso_select: Draws 500 random points, and lets you lasso select a subset. When selection changes, JavaScript computes the mean y value of selected points and updates a horizontal mean line.

In [19]:
from random import random

# Generate data 
x = [random() for _ in range(500)]
y = [random() for _ in range(500)]
s = ColumnDataSource(data=dict(x=x, y=y))


bg = "#eeeeee"           # soft gray background
gridc = "#cfcfcf"        # light grid
point_fill = "#9ea7a7"   # gray fill
point_edge = "#1b7f7a"   # teal outline
highlight_color = "#ff3b30"  # bright red
mean_line_color = "#f5a623"  # warm orange

# Figure 
p = figure(width=500, height=500,
           tools="lasso_select",
           title="Select Here",
           background_fill_color=bg,
           border_fill_color=bg)

p.grid.grid_line_color = gridc
p.grid.grid_line_alpha = 0.7

# Thicker teal border like your screenshot 
p.outline_line_width = 6
p.outline_line_color = "#1b7f7a"

# Scatter points 
p.scatter('x', 'y',
          source=s,
          size=10,
          fill_color=point_fill,
          line_color=point_edge,
          line_width=1.5,
          fill_alpha=0.6,
          selection_fill_color=highlight_color,
          selection_line_color=highlight_color,
          selection_fill_alpha=0.95,
          nonselection_fill_alpha=0.2,
          nonselection_line_alpha=0.2
         )
#This is why you get that “highlight subset” effect.


# Mean line source 
s2 = ColumnDataSource(data=dict(x=[0, 1], ym=[0.5, 0.5]))

p.line(x='x', y='ym',
       source=s2,
       color=mean_line_color,
       line_width=6,
       alpha=0.7)

# JS callback to update mean line 
s.selected.js_on_change('indices', CustomJS(args=dict(s=s, s2=s2), code="""
    const inds = s.selected.indices
    if (inds.length > 0) {
        const ym = inds.reduce((a, b) => a + s.data.y[b], 0) / inds.length
        s2.data = { x: s2.data.x, ym: [ym, ym] }
    }
"""))

output_notebook()
show(p)

Loading BokehJS ...

## Rating
### Package: Bokeh

#### Easiness: ★★★★★
#### Documentation: ★★★★★
#### Flexibility: ★★★★☆ 
#### Maturity: ★★★★★

##### Best for: [interactivty,visualization]
##### Avoid if: [you want a Completely static interface]


#### Overall Score: 95%